# Tutorial: Causal Models (CM) in aGrUM (C++)

Ported from the [pyAgrum do-calculus notebook](../../wrappers/pyagrum/doc/sphinx/notebooks/64-Causality_DoCalculusExamples.ipynb). A `CausalModel` wraps an *observational* `BayesNet` plus a declaration of which variables share unobserved (latent) common causes -- confounders that the BN's arcs alone don't show. Same caveat as the other C++ tutorials here: each `%%cpp` cell is a standalone program, no state shared with the previous cell.

In [1]:
%load_ext gum_cppnb

## Building a causal model with a latent confounder

`w->x->z->y;w->z`, with `x` and `y` additionally sharing an unobserved common cause (declared as a `LatentDescriptorVector`: a name plus the `NodeId`s of the variables it confounds -- `BayesNet::ids(...)` converts variable names to `NodeId`s). `CausalImpact` answers "is $P(y \mid do(x))$ identifiable from the observed data, and if so what's the formula/value".

In [2]:
%%cpp
#include <agrum/cm.h>

int main() {
  auto bn = gum::BayesNet<double>::fastPrototype("w->x->z->y;w->z");
  bn.generateCPTs();

  gum::CausalModel<double> cm(bn, {{"lat1", bn.ids({"x", "y"})}});

  gum::CausalImpact<double> ci(cm, {"y"}, {"x"});
  std::cout << "identified? " << (ci.isIdentified() ? "yes" : "no") << std::endl;
  std::cout << ci.toLatex() << std::endl;
  std::cout << ci.eval() << std::endl;
}

identified? yes
P\left(y \mid do(x)\right) = \sum_{w,z}{P\left(z\mid w,x\right) \cdot P\left(w\right) \cdot \left(\sum_{x'}{P\left(y\mid z\right) \cdot P\left(x'\mid w\right)}\right)}

      ║  y                │
x     ║0        │1        │
──────║─────────│─────────│
0     ║ 0.6564  │ 0.3436  │
1     ║ 0.7306  │ 0.2694  │



## The free `causalImpact()` helper

A convenience wrapper around the same identification algorithm, returning a `(CausalImpact, Tensor, explanation)` tuple directly -- no need to construct `CausalImpact` yourself first.

In [3]:
%%cpp
#include <agrum/cm.h>

int main() {
  auto bn = gum::BayesNet<double>::fastPrototype("w->x->z->y;w->z");
  bn.generateCPTs();
  gum::CausalModel<double> cm(bn, {{"lat1", bn.ids({"x", "y"})}});

  auto [formula, tensor, explanation] = gum::causalImpact<double>(cm, {"y"}, {"x"});
  std::cout << "explanation: " << explanation << std::endl;
  std::cout << tensor << std::endl;
}

explanation: Identified via do-calculus (ID/IDC).

      ║  y                │
x     ║0        │1        │
──────║─────────│─────────│
0     ║ 0.6564  │ 0.3436  │
1     ║ 0.7306  │ 0.2694  │



## Backdoor criterion

`CausalModel::backDoor(cause, effect)` looks for a set of observed variables satisfying Pearl's backdoor criterion -- here the confounder `z` between `x` and `y`.

In [4]:
%%cpp
#include <agrum/cm.h>

int main() {
  auto bn = gum::BayesNet<double>::fastPrototype("z->x->y;z->y");
  bn.generateCPTs();
  gum::CausalModel<double> cm(bn);   // fully observed DAG, no latent

  auto bd = cm.backDoor("x", "y");
  if (bd.has_value()) {
    std::cout << "backdoor set for (x,y), size " << bd->size() << ": ";
    for (auto n : *bd) std::cout << bn.variable(n).name() << " ";
    std::cout << std::endl;
  } else {
    std::cout << "no backdoor set found" << std::endl;
  }
}

backdoor set for (x,y), size 1: z 


## Unidentifiability (Hedge)

Not every causal effect is identifiable from observational data. When it isn't, `isIdentified()` returns `false` and `eval()`/`root()` throw `gum::OperationNotAllowed` -- the C++ equivalent of Python's `gum.HedgeException` (internally caught and turned into this "no identified formula" state, rather than propagating a `HedgeException` itself).

In [5]:
%%cpp
#include <agrum/cm.h>

int main() {
  auto bn = gum::BayesNet<double>::fastPrototype("X->Y;U");
  bn.generateCPTs();

  // X, Y and the unrelated U all share one latent common cause: classic unidentifiable case
  gum::LatentDescriptorVector lat;
  lat.emplace_back("Z", std::vector<gum::NodeId>{bn.idFromName("X"), bn.idFromName("Y"),
                                                  bn.idFromName("U")});
  gum::CausalModel<double> cm(bn, lat, /*assumeNonSpurious=*/true);

  gum::CausalImpact<double> ci(cm, {"Y"}, {"X"});
  std::cout << "identified? " << (ci.isIdentified() ? "yes" : "no") << std::endl;

  try {
    (void)ci.eval();
    std::cout << "eval() succeeded (unexpected)" << std::endl;
  } catch (gum::OperationNotAllowed&) {
    std::cout << "eval() threw OperationNotAllowed, as expected" << std::endl;
  }
}

identified? no
eval() threw OperationNotAllowed, as expected


## Counterfactual queries

"Given what we actually observed (`profile`), what would `on` have been if `whatif` had taken a different value?" -- `gum::Counterfactual` builds the corresponding twin network and evaluates it. `profile`/`values` are `HashTable<std::string,std::string>` mapping variable names to value **labels** (not indices).

In [6]:
%%cpp
#include <agrum/cm.h>

int main() {
  auto bn = gum::BayesNet<double>::fastPrototype("smoke->tar->cancer");
  bn.generateCPTs();
  gum::CausalModel<double> cm(bn);

  gum::HashTable<std::string, std::string> profile;
  profile.insert("smoke", "1");
  profile.insert("tar", "1");
  profile.insert("cancer", "1");

  gum::HashTable<std::string, std::string> values;
  values.insert("smoke", "0");   // what if this person hadn't smoked?

  gum::Counterfactual cf(cm, {"cancer"}, {"smoke"}, profile, values);
  std::cout << cf.toString() << std::endl;
  std::cout << "P(cancer | had-not-smoked, profile):\n" << cf.value() << std::endl;
}

[Counterfactual]
 on = {cancer}
 whatif = {smoke}
 profile: smoke=1, tar=1, cancer=1
 values: smoke=0
 result (symbolic): P(cancer|smoke)
 value (adapted to original variables):

  cancer           │
0        │1        │
─────────│─────────│
 0.5768  │ 0.4232  │


P(cancer | had-not-smoked, profile):

  cancer           │
0        │1        │
─────────│─────────│
 0.5768  │ 0.4232  │

